# Stacks, Queues, Deques & Heaps — Combined Masterclass

The seventh notebook in the DSA series. These four structures are **all variants of arrays** with different access patterns — but each one unlocks a class of problems that arrays alone cannot solve in linear time. The whole point of this notebook is internalizing **when to reach for each one and why it beats the array baseline.**

Out of scope: linked lists (separate notebook), trees, graphs, recursion / DP / backtracking. Sorting only where it crosses heap.

## How to read this notebook
1. **Section 1** — definitions, complexity table, and the "why not just use an array?" framing for each structure.
2. **Sections 2-3** — stack: implementation in Python (with traps), classic stack problems (balanced parens, infix→postfix, evaluation, min-stack).
3. **Sections 4-5** — monotonic stack: the single most-asked stack pattern. Templates for NGE / NSE / PGE / PSE, then the harder problems (stock span, daily temperatures, largest rectangle in histogram, trapping rain water).
4. **Sections 6-7** — queue: implementation (`deque` and "queue from 2 stacks" / "stack from 2 queues"), BFS framework, circular tour.
5. **Section 8** — deque and monotonic deque: sliding window max/min, constrained subset sum DP optimization.
6. **Sections 9-11** — heap: foundations including the **O(n) build-heap proof**, the full `heapq` library reference, heap sort.
7. **Sections 12-16** — the 5 master heap patterns: Top-K, K-Way Merge, Two Heaps, Scheduling, Sorted-Matrix-Kth.
8. **Section 17** — decision matrix: which DS for which signal.
9. **Section 18** — synthesis: complexity table, interview narration, common bugs.

## The "why not just an array?" cheat sheet

| Want to do … | Array cost | Better DS | New cost | Why |
|---|---|---|---|---|
| Last-in-first-out access | O(1) push/pop end | **Stack** (same) | O(1) | Just an array with restricted API; the *discipline* is what unlocks recursion-like algorithms |
| First-in-first-out access | O(n) pop front (`list.pop(0)`) | **Queue (`deque`)** | O(1) both ends | `list.pop(0)` shifts every element — disastrous in BFS loops |
| Find next greater/smaller in O(1) | O(n²) brute force | **Monotonic stack** | O(n) amortized | Each element pushed/popped at most once |
| Max/min in every sliding window | O(n·k) | **Monotonic deque** | O(n) | Stale elements never re-examined |
| Top-K out of N elements | O(n log n) sort | **Heap of size K** | O(n log K) | Smaller heap = cheaper log |
| Repeatedly extract min/max | O(n) per extract | **Heap** | O(log n) | Tree structure on array |
| Running median of a stream | O(n²) naive | **Two heaps** | O(log n) insert, O(1) query | Split halves: max-heap below, min-heap above |
| Merge K sorted lists | O(NK) brute or O(N log N) sort all | **Min-heap of size K** | O(N log K) | Always pull the global min in log K |

If you can recite this table from memory, you have the topic.

## The 8 master patterns

| # | Pattern | Canonical problem | DS |
|---|---|---|---|
| 1 | Bracket / matching | Valid parentheses | Stack |
| 2 | Expression / parser | Infix→postfix, postfix eval | Stack |
| 3 | Min-stack / max-stack | Get-min-in-O(1) | Stack with auxiliary |
| 4 | Monotonic stack | NGE, daily temps, largest rectangle, trap rain water | Stack |
| 5 | BFS / level order | Word Ladder, rotting oranges | Queue |
| 6 | Monotonic deque | Sliding window max/min | Deque |
| 7 | Top-K | Kth largest, top k frequent, k closest points | Heap of size K |
| 8 | K-way merge / Two heaps / Scheduling | Merge k lists, median stream, task scheduler | Heap(s) |


## 1. Foundations — definitions, complexities, and the "why not array" framing

### Stack — LIFO (Last In First Out)

Picture: a stack of plates. You add and remove from the **top**.

```
Push 1,2,3 then pop:
   |   |     |   |     |   |     | 3 |    | 3 |
   |   |     |   |     | 2 |     | 2 |    |   |
   |   |  →  | 1 |  →  | 1 |  →  | 1 |  → | 1 |
   ─────     ─────     ─────     ─────    ─────
            push 1   push 2,3  (state)   pop → 3
```

**Operations.** push, pop, peek/top, is_empty, size — all O(1).

**Why not just an array?** You CAN use one — `list.append()` and `list.pop()` are both O(1) amortized in Python. The "stack as a class" is a discipline contract: only top access. The wrappers exist for clarity in interview code.

**When the stack pattern matters:** recursion (the call stack IS a stack), parenthesis matching, expression evaluation, monotonic relationships, DFS, undo/redo.

### Queue — FIFO (First In First Out)

Picture: a checkout line. You join at the back, leave from the front.

**Operations.** enqueue (back), dequeue (front), peek_front, is_empty — all O(1) **if implemented correctly.**

**Why NOT a plain Python list?** Because `list.pop(0)` is **O(n)** — it shifts every remaining element left. If you do BFS on a list as a queue with `pop(0)`, you've turned an O(n) algorithm into O(n²). Always use `collections.deque` for queues — both ends are O(1).

### Deque — Double-Ended Queue

Picture: a queue that also lets you push back to the front (regret cancellation).

**Operations.** append, appendleft, pop, popleft — all O(1) in `collections.deque`.

**The Python deque is a doubly-linked list of fixed-size blocks.** That's why both ends are O(1) but random middle access is O(n) (unlike a list which is O(1) random access).

**Why a deque?** Sliding-window-max-class problems where you need to remove elements from both ends in O(1).

### Heap (Priority Queue) — extreme-element-first

Picture: a tournament bracket where the winner (smallest in a min-heap) sits at the top, and the heap auto-rebalances on every insert/extract.

A **binary heap** is a complete binary tree (all levels full except possibly the last, which is filled left-to-right) that satisfies the **heap property**:
- **Min-heap:** every parent ≤ its children.
- **Max-heap:** every parent ≥ its children.

Because it's a complete tree, we can store it in an array (no pointers needed):
- parent(i) = (i - 1) // 2
- left(i) = 2*i + 1
- right(i) = 2*i + 2

**Operations.**
- peek (min/max): O(1)
- insert (push): O(log n)
- extract-min/max (pop): O(log n)
- decrease-key / increase-key: O(log n)
- delete-arbitrary: O(log n) (if you have the index)
- **build_heap from array: O(n)** ← surprising; we prove this in §9
- heap sort: O(n log n)

**Why not just sort the array once and read off the top?** Because if data is **streaming** or you need only the **top K** out of N, you don't want O(N log N). Heap of size K gives you O(N log K) which is dramatically faster when K << N.

### Master complexity table

| Operation | Stack | Queue (deque) | Deque | Heap | Sorted array | Hash set |
|---|---|---|---|---|---|---|
| Peek extreme | O(1) (top only) | O(1) (front only) | O(1) (both ends) | O(1) | O(1) | — |
| Insert | O(1) | O(1) | O(1) | O(log n) | O(n) (insert sorted) | O(1) |
| Remove extreme | O(1) | O(1) | O(1) | O(log n) | O(1) | — |
| Find arbitrary | O(n) | O(n) | O(n) | O(n) | O(log n) | O(1) |
| Maintain order | LIFO | FIFO | both | extreme-first | full sort | none |

The heap is the only structure that gives you O(log n) for the extreme element on a *changing* set. That's its niche.


## 2. Stack — Python implementation

Three ways to get a stack in Python:

1. **A plain `list`** — `append()` to push, `pop()` to pop the top. Both O(1) amortized. **This is what 95% of interview code does** because it's the least ceremony.
2. **`collections.deque`** — `append()` / `pop()` are O(1) (not amortized — strictly O(1) since deque doesn't reallocate the way list does). Slightly faster in practice for push-heavy workloads, but for typical interview problems it's a wash.
3. **A class wrapper** — write a `Stack` class with explicit `push`/`pop`/`peek`. Useful when the interviewer asks "implement a stack" or when you want a `min-stack` variant with extra invariants.

In every problem below, we use a plain list as the stack unless the problem demands more.


In [ ]:
# Stack via list — the idiom you'll use 95% of the time
stack = []
stack.append(1)   # push
stack.append(2)
stack.append(3)
print("top:", stack[-1])           # peek
print("pop:", stack.pop())         # 3
print("size:", len(stack))         # 2
print("empty:", len(stack) == 0)   # False


In [ ]:
# Class wrapper version (use only when explicitly asked)
class Stack:
    def __init__(self):
        self._data = []
    def push(self, x):      self._data.append(x)
    def pop(self):          return self._data.pop() if self._data else None
    def peek(self):         return self._data[-1] if self._data else None
    def is_empty(self):     return not self._data
    def __len__(self):      return len(self._data)
    def __repr__(self):     return f"Stack({self._data})"

s = Stack()
for x in [1, 2, 3]: s.push(x)
assert s.peek() == 3
assert s.pop() == 3
assert len(s) == 2
print("Stack class works:", s)


### The big stack traps

1. **Forgetting to check `is_empty` before `peek`/`pop`** — Python's `list.pop()` on an empty list raises `IndexError`. Always guard.
2. **Confusing `pop()` with `pop(0)`** — `pop()` is the top (O(1)); `pop(0)` is the front (O(n)). The latter turns your stack into a slow queue.
3. **Using a stack when a queue is needed** — if order of arrival matters (e.g., BFS), stack will give you DFS-like behavior.


## 3. Classic stack patterns

### 3.1 Balanced Parentheses (LC 20)

Process left to right. On opening bracket, push. On closing bracket, the top of the stack must be its matching opening. At the end, the stack must be empty.

**Time O(n), Space O(n).**

This is the canonical "matching pairs" problem — the trick generalizes to HTML tag validation, JSON parens, etc.


In [ ]:
def is_valid_parens(s):
    pairs = {')': '(', ']': '[', '}': '{'}
    stack = []
    for c in s:
        if c in '([{':
            stack.append(c)
        elif c in ')]}':
            if not stack or stack[-1] != pairs[c]:
                return False
            stack.pop()
    return not stack

assert is_valid_parens("()") == True
assert is_valid_parens("()[]{}") == True
assert is_valid_parens("(]") == False
assert is_valid_parens("([)]") == False
assert is_valid_parens("{[]}") == True
assert is_valid_parens("") == True
print("is_valid_parens: OK — O(n) time")


### 3.2 Min Stack (LC 155) — get min in O(1)

A stack with `push`, `pop`, `top`, AND `get_min` all in **O(1).**

**Approach A: auxiliary stack.** Keep a parallel `min_stack` that holds the running min at each level. On push, push `min(x, min_stack[-1])` onto min_stack. On pop, pop both.

**Approach B (your reference's clever trick): single stack with delta encoding.** When pushing a value `x < current_min`, push `2*x - current_min` (or `current_min - x` if you store the delta), then update current_min. On pop, if the popped value is less than current_min, restore current_min from the delta. **O(1) extra space**, but only works cleanly for positive numbers and requires careful arithmetic.

In interviews use Approach A unless asked for O(1) space. It's clearer and always correct.

**Time O(1) for all ops, Space O(n) — Approach A** | **Time O(1), Space O(1) auxiliary — Approach B.**


In [ ]:
# Approach A — auxiliary stack (recommended in interviews)
class MinStack:
    def __init__(self):
        self.stack = []
        self.min_stack = []   # parallel running-min stack
    def push(self, x):
        self.stack.append(x)
        if not self.min_stack or x <= self.min_stack[-1]:
            self.min_stack.append(x)
        else:
            self.min_stack.append(self.min_stack[-1])   # keep length aligned
    def pop(self):
        self.min_stack.pop()
        return self.stack.pop()
    def top(self):
        return self.stack[-1]
    def get_min(self):
        return self.min_stack[-1]

ms = MinStack()
for x in [3, 1, 5, 2, 1]:
    ms.push(x)
assert ms.get_min() == 1; ms.pop()    # remove the last 1
assert ms.get_min() == 1              # other 1 still there
ms.pop(); ms.pop()                    # remove 2, 5
assert ms.get_min() == 1; ms.pop()    # remove 1
assert ms.get_min() == 3              # only 3 left
print("MinStack (aux stack): OK — all ops O(1)")


In [ ]:
# Approach B — O(1) auxiliary space using delta encoding.
# Works for any integers (positive or negative).
# Stored value = x when x >= curr_min, else x is encoded relative to old min.
# Encoding: when pushing x < curr_min, push (2*x - curr_min), then set curr_min = x.
class MinStackO1:
    def __init__(self):
        self.stack = []
        self.curr_min = None
    def push(self, x):
        if not self.stack:
            self.stack.append(x)
            self.curr_min = x
        elif x >= self.curr_min:
            self.stack.append(x)
        else:
            self.stack.append(2 * x - self.curr_min)   # encoded
            self.curr_min = x
    def pop(self):
        top = self.stack.pop()
        if top < self.curr_min:                      # was an encoded value
            old_min = 2 * self.curr_min - top         # decode
            popped = self.curr_min
            self.curr_min = old_min if self.stack else None
            return popped
        if not self.stack: self.curr_min = None
        return top
    def top(self):
        top = self.stack[-1]
        return self.curr_min if top < self.curr_min else top
    def get_min(self):
        return self.curr_min

ms = MinStackO1()
for x in [3, 1, 5, 2, 1, -7]:
    ms.push(x)
assert ms.get_min() == -7
ms.pop()
assert ms.get_min() == 1
print("MinStackO1: OK — O(1) extra space using delta encoding")


### 3.3 Infix → Postfix (Shunting-yard)

Postfix (Reverse Polish) form has no parentheses and no precedence ambiguity. Convert via a stack of operators:

- **Operands** → directly append to output.
- **Operators**: while the stack top has **higher-or-equal precedence**, pop it to output, then push the new operator.
- **`(`** → push onto operator stack.
- **`)`** → pop operators to output until `(` is removed.
- **End of input** → drain stack to output.

**Precedence:** `^` (right-assoc, 3) > `* / %` (2) > `+ -` (1). For right-associative operators (only `^` typically) you compare strictly greater rather than greater-equal — but for interview code, `^` rarely shows up; default to left-assoc rules and mention the caveat.

**Time O(n), Space O(n).**


In [ ]:
def infix_to_postfix(expr):
    prec = {'+': 1, '-': 1, '*': 2, '/': 2, '%': 2, '^': 3}
    stack = []
    out = []
    for ch in expr:
        if ch.isalnum():
            out.append(ch)
        elif ch == '(':
            stack.append(ch)
        elif ch == ')':
            while stack and stack[-1] != '(':
                out.append(stack.pop())
            stack.pop()                              # discard the '('
        else:                                        # an operator
            while stack and stack[-1] != '(' and prec.get(stack[-1], 0) >= prec[ch]:
                out.append(stack.pop())
            stack.append(ch)
    while stack:
        out.append(stack.pop())
    return ''.join(out)

assert infix_to_postfix("a+b*c") == "abc*+"
assert infix_to_postfix("(a+b)*c") == "ab+c*"
assert infix_to_postfix("a+b*(c^d-e)^(f+g*h)-i") == "abcd^e-fgh*+^*+i-"
print("infix_to_postfix: OK")


### 3.4 Evaluate Postfix / Reverse Polish (LC 150)

For each token: if operand, push; if operator, pop two operands, apply, push the result. The final stack top is the answer.

**Critical detail:** the operand order is `val1 (op) val2` where `val2 = stack.pop()` first, `val1 = stack.pop()` second. Swap them and `5 - 3` becomes `3 - 5`. Burn this in.

**Time O(n), Space O(n).**


In [ ]:
def eval_RPN(tokens):
    stack = []
    ops = {'+', '-', '*', '/'}
    for t in tokens:
        if t in ops:
            b = stack.pop()                # SECOND popped is the right operand
            a = stack.pop()                # FIRST popped (from the bottom) is the left
            if t == '+': stack.append(a + b)
            elif t == '-': stack.append(a - b)
            elif t == '*': stack.append(a * b)
            else:        stack.append(int(a / b))   # LC truncates toward 0, not floor
        else:
            stack.append(int(t))
    return stack[0]

assert eval_RPN(["2","1","+","3","*"]) == 9
assert eval_RPN(["4","13","5","/","+"]) == 6
assert eval_RPN(["10","6","9","3","+","-11","*","/","*","17","+","5","+"]) == 22
print("eval_RPN: OK")


**Why `int(a/b)` and not `a//b`?** Because `//` in Python is **floor division**, which rounds toward negative infinity. LC 150 specifies **truncation toward zero**. So `-7 // 2 == -4` (wrong), but `int(-7/2) == -3` (right). This is a notorious one-line trap in LeetCode submissions.

### 3.5 Valid Stack Sequences (LC 946)

Given two arrays `pushed` and `popped`, decide if `popped` could result from a sequence of pushes and pops on an empty stack with `pushed` in order.

**Simulation.** Walk `pushed`. After each push, while stack-top equals `popped[j]`, pop and advance j. At the end, valid iff stack is empty.

**Time O(n), Space O(n).**


In [ ]:
def validate_stack_sequences(pushed, popped):
    stack, j = [], 0
    for x in pushed:
        stack.append(x)
        while stack and j < len(popped) and stack[-1] == popped[j]:
            stack.pop()
            j += 1
    return not stack

assert validate_stack_sequences([1,2,3,4,5], [4,5,3,2,1]) == True
assert validate_stack_sequences([1,2,3,4,5], [4,3,5,1,2]) == False
print("validate_stack_sequences: OK")


### Classic stack practice problems

| Concept | LeetCode |
|---------|----------|
| Valid Parentheses | LC 20 |
| Min Stack | LC 155 |
| Evaluate Reverse Polish | LC 150 |
| Basic Calculator | LC 224 |
| Basic Calculator II | LC 227 |
| Decode String (nested k[chars]) | LC 394 |
| Asteroid Collision | LC 735 |
| Remove All Adjacent Duplicates | LC 1047 |
| Remove K Digits | LC 402 |
| Simplify Path (unix-style) | LC 71 |
| Validate Stack Sequences | LC 946 |
| Backspace String Compare | LC 844 |


## 4. Monotonic stack — the single most important stack pattern

This is the topic interviewers love because it turns O(n²) brute force into O(n) elegance, and the trick isn't obvious.

### What a monotonic stack is
A stack where elements maintain a **monotonic** (always increasing OR always decreasing) order from bottom to top. Before pushing a new element, you **pop** all elements that would violate the order. Each element is pushed and popped **at most once** → total **O(n)** work.

### The decoder ring — which monotonic order solves which problem

| Want to find | Stack contains | Direction | Pop rule |
|---|---|---|---|
| **Next Greater Element** | **Decreasing** values (or indices) | Left → right | While stack top **≤** curr, pop and record curr as their NGE |
| **Next Smaller Element** | **Increasing** values | Left → right | While stack top **≥** curr, pop and record curr as their NSE |
| **Previous Greater Element** | **Decreasing** values | Left → right | While stack top **≤** curr, pop. After popping, the new top (if any) is curr's PGE. Push curr. |
| **Previous Smaller Element** | **Increasing** values | Left → right | Same as PGE but flipped comparison |

**The two patterns of consumption.**

- **"Find next" style** — process curr, the elements **popped** are the ones for which curr is their answer.
- **"Find previous" style** — process curr, the element **remaining on top** after popping is curr's answer.

Memorize: **"Pop when violated, remaining top is the answer."** That single sentence is the entire monotonic-stack mental model.

### Why monotonic = O(n)?
Each element enters the stack once and leaves at most once. So across all of the outer loop's iterations, the inner while loop does at most n total iterations. **Amortized O(1) per element**, total O(n).


In [ ]:
# === TEMPLATE: Next Greater Element ===
# Returns nge[i] = the first element to the right of i greater than nums[i], else -1.
def next_greater(nums):
    n = len(nums)
    nge = [-1] * n
    stack = []                                  # store INDICES, not values
    for i in range(n):
        while stack and nums[stack[-1]] < nums[i]:
            nge[stack.pop()] = nums[i]          # i is the NGE for the popped index
        stack.append(i)
    return nge

assert next_greater([2,1,3,2,4,3]) == [3,3,4,4,-1,-1]
assert next_greater([4,5,2,25]) == [5,25,25,-1]
print("next_greater: OK — O(n)")


In [ ]:
# === TEMPLATE: Previous Smaller Element ===
# Returns pse[i] = the closest element to the LEFT of i that is strictly smaller, else -1.
def previous_smaller(nums):
    n = len(nums)
    pse = [-1] * n
    stack = []
    for i in range(n):
        # Pop while the top is NOT smaller than nums[i]
        while stack and nums[stack[-1]] >= nums[i]:
            stack.pop()
        if stack:
            pse[i] = nums[stack[-1]]            # whoever's still on top
        stack.append(i)
    return pse

assert previous_smaller([1, 3, 2, 4]) == [-1, 1, 1, 2]
assert previous_smaller([5, 4, 3, 2, 1]) == [-1, -1, -1, -1, -1]
print("previous_smaller: OK — O(n)")


### The "all four in one pass" trick

For largest-rectangle-style problems you sometimes need **both** previous-smaller and next-smaller. Instead of two separate passes, do it in **one** pass: when you pop an element, the index that's about to be popped has:
- Its **next smaller** = current index (`i`) — because that's what triggered the pop.
- Its **previous smaller** = the new stack top (the element below it after popping).

This is the technique used in §4.4 below.


## 5. Monotonic stack problems

### 5.1 Daily Temperatures (LC 739)

For each day, how many days until a warmer one? This is "next greater element, indexed by distance."

**Time O(n), Space O(n).**


In [ ]:
def daily_temperatures(temps):
    n = len(temps)
    ans = [0] * n
    stack = []                          # indices of days awaiting their warmer day
    for i in range(n):
        while stack and temps[stack[-1]] < temps[i]:
            j = stack.pop()
            ans[j] = i - j              # distance to the warmer day
        stack.append(i)
    return ans

assert daily_temperatures([73,74,75,71,69,72,76,73]) == [1,1,4,2,1,1,0,0]
assert daily_temperatures([30,40,50,60]) == [1,1,1,0]
print("daily_temperatures: OK")


### 5.2 Stock Span (GFG / LC 901)

The span of stock price on day i = number of consecutive days **immediately preceding** day i (and including i) where the price was ≤ price[i]. Equivalently: `span[i] = i - PGE_index[i]`, where PGE is "previous greater" (strictly greater).

**Why a monotonic decreasing stack?** We want the previous greater element. Maintain a decreasing stack: when the new price is **higher** than top, pop. The element remaining on top (if any) is the previous greater day.


In [ ]:
def stock_span(prices):
    n = len(prices)
    spans = [0] * n
    stack = []                              # indices, prices monotonically decreasing
    for i in range(n):
        while stack and prices[stack[-1]] <= prices[i]:
            stack.pop()
        spans[i] = i + 1 if not stack else i - stack[-1]
        stack.append(i)
    return spans

assert stock_span([100, 80, 60, 70, 60, 75, 85]) == [1, 1, 1, 2, 1, 4, 6]
assert stock_span([10, 20, 30, 40, 50]) == [1, 2, 3, 4, 5]
assert stock_span([5, 4, 3, 2, 1]) == [1, 1, 1, 1, 1]
print("stock_span: OK — O(n) amortized")


### 5.3 Next Greater Element II — Circular Array (LC 503)

The array wraps around: element at index n-1 has element 0 as its "next." The trick: iterate `2n` times using `i % n`. After the first n iterations, we've populated NGE for every element that has one within the original array; the second pass handles wrap-around.

**Time O(n), Space O(n).**


In [ ]:
def next_greater_circular(nums):
    n = len(nums)
    ans = [-1] * n
    stack = []
    for i in range(2 * n):
        cur = nums[i % n]
        while stack and nums[stack[-1]] < cur:
            ans[stack.pop()] = cur
        if i < n:                           # only push original indices
            stack.append(i)
    return ans

assert next_greater_circular([1,2,1]) == [2,-1,2]
assert next_greater_circular([1,2,3,4,3]) == [2,3,4,-1,4]
print("next_greater_circular: OK")


### 5.4 Largest Rectangle in Histogram (LC 84) — THE canonical monotonic stack problem

You have bars of given heights, each of width 1. Find the area of the largest rectangle that fits inside the histogram.

**The insight.** For each bar `i`, the largest rectangle **using bar i as the limiting height** extends left to the previous-smaller bar and right to the next-smaller bar:
```
width_i = (next_smaller_index_i - prev_smaller_index_i - 1)
area_i  = heights[i] * width_i
```

So: compute PSE and NSE for every bar, sweep through. But the **one-pass** version is more elegant:

- Maintain an **increasing** monotonic stack of indices.
- When `heights[i] < heights[stack[-1]]`, the bar at `stack[-1]` has just found its next-smaller (it's `i`). Its previous-smaller is the index now below the popped one (or -1 if stack empty). Compute its area and update max.
- After the main loop, drain the remaining stack with a virtual sentinel at index `n` (smaller-than-everything).

This is the most beautiful O(n) algorithm in interview-land. Internalize it.

**Time O(n), Space O(n).**


In [ ]:
def largest_rectangle_area(heights):
    stack = []                                # indices of bars in strictly increasing height
    max_area = 0
    n = len(heights)
    for i in range(n + 1):
        # Sentinel of height 0 at i == n forces all remaining bars to be popped
        cur_h = 0 if i == n else heights[i]
        while stack and heights[stack[-1]] >= cur_h:
            h = heights[stack.pop()]
            # Previous-smaller is now the new stack top (or -1 if empty)
            width = i if not stack else i - stack[-1] - 1
            max_area = max(max_area, h * width)
        stack.append(i)
    return max_area

assert largest_rectangle_area([2,1,5,6,2,3]) == 10           # 5 and 6 → 5 * 2 = 10
assert largest_rectangle_area([6, 2, 5, 4, 1, 5, 6]) == 10   # best is 5*2=10 (bars at idx 5,6)
assert largest_rectangle_area([2, 4]) == 4
assert largest_rectangle_area([2, 5, 1]) == 5
print("largest_rectangle_area: OK — O(n) one-pass")


### 5.5 Maximal Rectangle (LC 85) — extension of 5.4

Given a binary matrix, find the largest all-1 rectangle.

**Trick:** for each row, treat the running consecutive-1s ending at that row as histogram heights. Run §5.4 on each row. **Total time O(m·n).**


In [ ]:
def maximal_rectangle(matrix):
    if not matrix: return 0
    n = len(matrix[0])
    heights = [0] * n
    best = 0
    for row in matrix:
        for j in range(n):
            heights[j] = heights[j] + 1 if row[j] == "1" else 0
        best = max(best, largest_rectangle_area(heights))
    return best

mat = [["1","0","1","0","0"],
       ["1","0","1","1","1"],
       ["1","1","1","1","1"],
       ["1","0","0","1","0"]]
assert maximal_rectangle(mat) == 6
print("maximal_rectangle: OK — O(m·n)")


### 5.6 Trapping Rain Water (LC 42) — monotonic stack version

Each bar holds water bounded by the taller bars on either side. Standard solution is two pointers O(n), O(1), but the **monotonic stack solution** is the classic stack interview answer:

- Maintain a **decreasing** stack of indices.
- When we see `height[i] > height[stack[-1]]`, we've found a "right wall" for the popped bar. The left wall is the new stack top (after popping).
- Water trapped above the popped bar = `(min(left, right) - popped) * width`.

**Time O(n), Space O(n).** (Two-pointer is strictly better in space, but this version demonstrates the monotonic stack pattern.)


In [ ]:
def trap_rain_water(height):
    stack = []
    water = 0
    for i, h in enumerate(height):
        while stack and height[stack[-1]] < h:
            mid = stack.pop()
            if not stack:
                break
            left = stack[-1]
            width = i - left - 1
            bounded = min(height[left], h) - height[mid]
            water += width * bounded
        stack.append(i)
    return water

assert trap_rain_water([0,1,0,2,1,0,1,3,2,1,2,1]) == 6
assert trap_rain_water([4,2,0,3,2,5]) == 9
print("trap_rain_water: OK — monotonic stack O(n) time, O(n) space")


### 5.7 Sum of Subarray Minimums (LC 907) — contribution technique

For every subarray, sum up its minimum. Brute force is O(n²). The monotonic stack trick: for each element `nums[i]`, count **how many subarrays have nums[i] as their min** — that's `(i - PSE_left) * (NSE_right - i)`. Multiply by `nums[i]`, sum across all i.

**Time O(n), Space O(n).** "Contribution" thinking generalizes — also used in Sum of Subarray Ranges (LC 2104).

**Trap: handle duplicates** — to avoid double-counting subarrays whose minimum appears multiple times, use **strict** comparison on one side and **non-strict** on the other (e.g., PSE strict-less, NSE non-strict-less-or-equal). Otherwise a subarray where min appears twice will be counted twice.


In [ ]:
def sum_subarray_mins(nums):
    MOD = 10**9 + 7
    n = len(nums)
    # PSE: previous strictly smaller; NSE: next less-or-equal (asymmetric tie-break)
    left = [-1] * n
    right = [n] * n
    stack = []
    for i in range(n):
        while stack and nums[stack[-1]] >= nums[i]:    # >= → strictly smaller PSE
            stack.pop()
        if stack: left[i] = stack[-1]
        stack.append(i)
    stack = []
    for i in range(n-1, -1, -1):
        while stack and nums[stack[-1]] > nums[i]:     # > → less-or-equal NSE
            stack.pop()
        if stack: right[i] = stack[-1]
        stack.append(i)
    return sum(nums[i] * (i - left[i]) * (right[i] - i) for i in range(n)) % MOD

assert sum_subarray_mins([3,1,2,4]) == 17
assert sum_subarray_mins([11,81,94,43,3]) == 444
print("sum_subarray_mins: OK — O(n) contribution technique")


### Monotonic stack practice problems

| Concept | LeetCode |
|---------|----------|
| Next Greater Element I | LC 496 |
| Next Greater Element II (circular) | LC 503 |
| Daily Temperatures | LC 739 |
| Online Stock Span | LC 901 |
| Largest Rectangle in Histogram | LC 84 |
| Maximal Rectangle | LC 85 |
| Trapping Rain Water | LC 42 |
| Sum of Subarray Minimums | LC 907 |
| Sum of Subarray Ranges | LC 2104 |
| Number of Visible People in a Queue | LC 1944 |
| Maximum Subarray Min-Product | LC 1856 |
| Car Fleet | LC 853 (monotonic stack on sorted positions) |


## 6. Queue — implementation

**Use `collections.deque`.** Don't roll your own with a list. Don't even *think* about `list.pop(0)` in a queue. It's O(n) and silently turns BFS into O(n²) — the most common "why is my code TLE" bug in BFS problems.


In [ ]:
from collections import deque

q = deque()
q.append(1); q.append(2); q.append(3)
print("front:", q[0])                # peek front
print("dequeue:", q.popleft())       # 1
print("after:", list(q))             # [2, 3]
print("size:", len(q))
print("empty:", len(q) == 0)


### Queue from two stacks (LC 232)

Classic interview trick to test understanding. `in_stack` for enqueue, `out_stack` for dequeue. When dequeue and `out_stack` is empty, dump everything from `in_stack` (reversing the order).

**Amortized O(1) per operation** — each element is moved between stacks at most once.


In [ ]:
class QueueViaStacks:
    def __init__(self):
        self.in_stack = []
        self.out_stack = []
    def push(self, x):
        self.in_stack.append(x)
    def pop(self):
        if not self.out_stack:
            while self.in_stack:
                self.out_stack.append(self.in_stack.pop())
        return self.out_stack.pop()
    def peek(self):
        if not self.out_stack:
            while self.in_stack:
                self.out_stack.append(self.in_stack.pop())
        return self.out_stack[-1]
    def empty(self):
        return not self.in_stack and not self.out_stack

q = QueueViaStacks()
for x in [1,2,3,4]: q.push(x)
assert q.peek() == 1
assert q.pop() == 1
q.push(5)
assert q.pop() == 2
assert q.pop() == 3
print("QueueViaStacks: OK — amortized O(1)")


### Stack from two queues (LC 225)

Reverse direction. Two common approaches:

**Push-heavy:** On push, enqueue x then rotate the queue so x is at the front. `push` is O(n), `pop` is O(1).

**Pop-heavy:** Enqueue normally. On pop, transfer all but the last element to the second queue, then pop the last. `push` is O(1), `pop` is O(n).

Both are O(n) amortized for the heavy operation. Mention both in interviews.


In [ ]:
class StackViaQueues:
    # Push O(n), pop O(1) — using one queue.
    def __init__(self):
        self.q = deque()
    def push(self, x):
        self.q.append(x)
        # Rotate so x is at the front
        for _ in range(len(self.q) - 1):
            self.q.append(self.q.popleft())
    def pop(self):
        return self.q.popleft()
    def top(self):
        return self.q[0]
    def empty(self):
        return not self.q

s = StackViaQueues()
for x in [1, 2, 3]: s.push(x)
assert s.top() == 3
assert s.pop() == 3
assert s.pop() == 2
print("StackViaQueues: OK")


## 7. Queue patterns

### 7.1 BFS — the canonical queue pattern

Whenever a problem mentions "shortest path", "minimum steps", "level by level", or "spread out from a starting point", **BFS** is the answer, and BFS needs a queue.

The template every BFS follows:
```python
from collections import deque

def bfs(start):
    q = deque([start])
    visited = {start}
    steps = 0
    while q:
        for _ in range(len(q)):            # snapshot — process current level
            node = q.popleft()
            # ... process node ...
            for neighbor in neighbors(node):
                if neighbor not in visited:
                    visited.add(neighbor)
                    q.append(neighbor)
        steps += 1
    return result
```

The `for _ in range(len(q))` is the **"level snapshot"** trick — freeze the queue length at the start of the level so newly-added neighbors are processed in the next iteration, not this one. This is how you cleanly track distance from the source.

This template handles: shortest path in unweighted graph, level-order tree traversal, word ladder, rotting oranges, knight's shortest move, walls and gates, all-distances variants. Memorize it.

### 7.2 Rotting Oranges (LC 994) — multi-source BFS

Fresh orange rots if adjacent to a rotten one. Each minute, all neighbors of rotten oranges rot. Return minutes until no fresh remains (or -1).

**Trick: multi-source BFS.** Put **all** initially-rotten oranges into the queue at the start. Then run level-snapshot BFS. The answer is the number of levels - 1 (or 0 if there were no fresh to begin with).


In [ ]:
def oranges_rotting(grid):
    if not grid: return 0
    m, n = len(grid), len(grid[0])
    q = deque()
    fresh = 0
    for i in range(m):
        for j in range(n):
            if grid[i][j] == 2: q.append((i, j))
            elif grid[i][j] == 1: fresh += 1
    minutes = 0
    while q and fresh:
        for _ in range(len(q)):                          # level snapshot
            i, j = q.popleft()
            for di, dj in [(-1,0),(1,0),(0,-1),(0,1)]:
                ni, nj = i+di, j+dj
                if 0 <= ni < m and 0 <= nj < n and grid[ni][nj] == 1:
                    grid[ni][nj] = 2
                    fresh -= 1
                    q.append((ni, nj))
        minutes += 1
    return -1 if fresh else minutes

assert oranges_rotting([[2,1,1],[1,1,0],[0,1,1]]) == 4
assert oranges_rotting([[2,1,1],[0,1,1],[1,0,1]]) == -1
assert oranges_rotting([[0,2]]) == 0
print("oranges_rotting: OK — multi-source BFS")


### 7.3 First Negative Number in Every Window (Sliding Window + Queue)

Find the first negative number in every subarray of size k. Use a queue (or deque) of indices of negative numbers; advance the window by removing stale indices.

**Time O(n), Space O(k).**


In [ ]:
def first_negative_in_window(arr, k):
    out = []
    q = deque()                     # indices of negatives in the current window
    for i, x in enumerate(arr):
        if x < 0: q.append(i)
        if i >= k - 1:
            while q and q[0] <= i - k:
                q.popleft()
            out.append(arr[q[0]] if q else 0)
    return out

assert first_negative_in_window([-8, 2, 3, -6, 10], 2) == [-8, 0, -6, -6]
assert first_negative_in_window([12, -1, -7, 8, -15, 30, 16, 28], 3) == [-1, -1, -7, -15, -15, 0]
print("first_negative_in_window: OK")


### 7.4 First Circular Tour / Gas Station (LC 134)

There are N gas stations in a circle. `petrol[i]` is gas available at station i; `dist[i]` is the distance to the next. With each gas unit driving 1 distance unit, find the starting station that lets you complete the loop, else -1.

**Why is this a queue problem traditionally?** A queue-based simulation tracks "stations included so far"; on overflow, pop the front. But the **O(n) one-pass** insight beats this:

**Key observation.** If you run out of gas going from station 0 to station i, then **no station between 0 and i** can be a valid start. (Proof: if you ran out before reaching i, you'd run out even worse if you tried to start later, because you'd have less of the gas you accumulated up to i.)

So we reset the candidate start to `i + 1` whenever the running tank goes negative. Track total surplus to decide if any solution exists.

**Time O(n), Space O(1).**


In [ ]:
def first_circular_tour(petrol, dist):
    total_surplus = 0           # if negative at the end, no solution exists
    tank = 0
    start = 0
    for i in range(len(petrol)):
        gain = petrol[i] - dist[i]
        tank += gain
        total_surplus += gain
        if tank < 0:
            start = i + 1
            tank = 0
    return start if total_surplus >= 0 else -1

# 0-indexed return
assert first_circular_tour([4, 8, 7, 4], [6, 5, 3, 5]) == 1
assert first_circular_tour([4, 8], [5, 6]) == 1
assert first_circular_tour([8, 9, 4], [5, 10, 12]) == -1
print("first_circular_tour: OK — O(n), O(1)")


**Where's the queue here?** It's not actually needed for the optimal solution. The queue-based simulation is the *natural* first thought, but the surplus argument eliminates the queue entirely. This is a great interview narrative: "I'd start with a queue simulation that's O(n²) worst case, but here's an observation that drops it to O(n), O(1)."

### Queue practice problems

| Concept | LeetCode |
|---------|----------|
| Implement Queue using Stacks | LC 232 |
| Implement Stack using Queues | LC 225 |
| Design Circular Queue | LC 622 |
| Design Circular Deque | LC 641 |
| Rotting Oranges (multi-source BFS) | LC 994 |
| 01 Matrix | LC 542 (multi-source BFS) |
| Walls and Gates | LC 286 |
| Number of Islands (BFS version) | LC 200 |
| Word Ladder | LC 127 |
| Open the Lock | LC 752 |
| Perfect Squares (BFS framing) | LC 279 |
| First Negative in Every Window | GFG |
| Gas Station / Circular Tour | LC 134 |


## 8. Deque and monotonic deque

A **deque** (double-ended queue) supports O(1) operations on **both** ends. In Python, `collections.deque` is the standard implementation (a doubly-linked list of fixed-size blocks).

```
            ← popleft       append →
                 ←  appendleft   pop →
            [   a   b   c   d   e   ]
```

### When you actually need a deque (not just a stack or queue)
- **Sliding window max/min** — you need to remove stale indices from the front AND maintain a monotonic order from the back.
- **Palindrome check** — pop from both ends and compare.
- **Implementing both a stack and a queue with one structure** — but in practice, separating them is cleaner.

### The monotonic deque pattern — sliding window max/min

This is THE pattern that justifies the deque's existence in interview problems. **Same logic as a monotonic stack, but the deque lets us also evict stale window elements from the front in O(1).**

**Algorithm (for sliding window MAX):**
1. Maintain a deque of **indices** with values in **decreasing** order.
2. For each new index `i`:
   - **Evict from front** any indices that fell out of the window (`q[0] <= i - k`).
   - **Pop from back** while `nums[q[-1]] <= nums[i]` — those values can never be max again because nums[i] is bigger and stays in the window longer.
   - Append `i`.
3. Once `i >= k - 1`, the front of the deque is the max of the current window.

Every index pushed/popped at most once → **O(n) total, O(k) space.**


In [ ]:
def max_sliding_window(nums, k):
    q = deque()                          # indices, decreasing nums values
    out = []
    for i, x in enumerate(nums):
        # Step 1: evict front if out-of-window
        while q and q[0] <= i - k:
            q.popleft()
        # Step 2: maintain decreasing — pop back while smaller-or-equal
        while q and nums[q[-1]] <= x:
            q.pop()
        q.append(i)
        # Step 3: window is full → record max
        if i >= k - 1:
            out.append(nums[q[0]])
    return out

assert max_sliding_window([1,3,-1,-3,5,3,6,7], 3) == [3,3,5,5,6,7]
assert max_sliding_window([10, 8, 5, 12, 15, 7, 6], 3) == [10, 12, 15, 15, 15]
assert max_sliding_window([1], 1) == [1]
print("max_sliding_window: OK — O(n) time, O(k) space")


### 8.2 Sliding window minimum — symmetric

Same template, flip the monotonic order to **increasing**.


In [ ]:
def min_sliding_window(nums, k):
    q = deque()                          # indices, increasing nums values
    out = []
    for i, x in enumerate(nums):
        while q and q[0] <= i - k: q.popleft()
        while q and nums[q[-1]] >= x: q.pop()
        q.append(i)
        if i >= k - 1: out.append(nums[q[0]])
    return out

assert min_sliding_window([1,3,-1,-3,5,3,6,7], 3) == [-1,-3,-3,-3,3,3]
print("min_sliding_window: OK")


### 8.3 Constrained Subsequence Sum (LC 1425) — DP optimized by monotonic deque

`dp[i]` = max sum of a subsequence ending at i, where consecutive picks must be at most k indices apart.

```
dp[i] = nums[i] + max(0, max(dp[j] for j in [i-k, i-1]))
```

The "max over a sliding window" inside the recurrence is exactly the monotonic deque pattern. This drops the DP from O(n·k) to **O(n)**.

This is THE classic example of "DP transition optimized by monotonic deque." A great senior-level interview topic.


In [ ]:
def constrained_subset_sum(nums, k):
    n = len(nums)
    dp = [0] * n
    q = deque()                          # indices of dp[j] with decreasing dp values
    for i, x in enumerate(nums):
        while q and q[0] < i - k: q.popleft()
        dp[i] = x + (dp[q[0]] if q and dp[q[0]] > 0 else 0)
        while q and dp[q[-1]] <= dp[i]: q.pop()
        q.append(i)
    return max(dp)

assert constrained_subset_sum([10,2,-10,5,20], 2) == 37
assert constrained_subset_sum([-1,-2,-3], 1) == -1
print("constrained_subset_sum: OK — DP + monotonic deque, O(n)")


### Deque practice problems

| Concept | LeetCode |
|---------|----------|
| Sliding Window Maximum | LC 239 |
| Sliding Window Minimum | LC 1004 variant |
| Design Circular Deque | LC 641 |
| Shortest Subarray with Sum at Least K | LC 862 |
| Constrained Subsequence Sum | LC 1425 |
| Jump Game VI | LC 1696 |
| Longest Continuous Subarray with Absolute Diff ≤ Limit | LC 1438 (deque for max AND min) |
| Number of Subarrays with Bounded Maximum | LC 795 |


## 9. Heap foundations

A **binary heap** is a complete binary tree (every level full except possibly the last, filled left-to-right) with the **heap property**:
- **Min-heap:** every parent ≤ both children.
- **Max-heap:** every parent ≥ both children.

Because the tree is complete, we store it in a flat array. For a node at index `i`:
- **parent**: `(i - 1) // 2`
- **left child**: `2 * i + 1`
- **right child**: `2 * i + 2`

### Visualization: array ↔ tree

```
Index:  0   1   2   3   4   5   6
Value:  1   3   2   7   8   4   5

                 1                 ← root (index 0)
               /   \
              3     2              ← indices 1, 2
             / \   / \
            7   8 4   5            ← indices 3,4,5,6
```

Check: every parent ≤ children. ✓ Min-heap.

### The two core operations: heapify-up and heapify-down

**heapify-up (sift-up).** Used after `insert` at the end. Walk up the tree, swapping with parent while smaller (min-heap). **O(log n)** — height of a complete binary tree on n nodes.

**heapify-down (sift-down).** Used after `extract` (which replaces root with last element). Walk down, swapping with the smaller child while larger. **O(log n).**

### Build-heap from an unordered array is O(n), NOT O(n log n) — and you should know why

The naive way: insert each element one at a time → n insertions × O(log n) = O(n log n).

The clever way: take the array as-is, then call **heapify-down on every internal node, starting from the bottom up.** Internal nodes are indices `0 .. n//2 - 1`; leaves are already trivially "heaps of size 1."

```python
for i in range(n // 2 - 1, -1, -1):
    heapify_down(i)
```

**Why is this O(n)?** Heapify-down's cost is proportional to the node's height (distance to a leaf), not depth. In a complete tree:
- Roughly n/2 leaves at height 0 (zero work)
- n/4 nodes at height 1 (1 unit of work)
- n/8 nodes at height 2 (2 units of work)
- ...

Total work = ∑ (n / 2^(h+1)) × h for h = 0 to log n. This sum converges to **O(n)**. (Bound it by ∑ h/2^h = 2 for the infinite series.)

This is **a famous interview question** — "Why is build_heap O(n) and not O(n log n)?" Have this answer rehearsed.

### Complexity reference

| Operation | Min-heap (binary) |
|-----------|-------------------|
| peek (min) | O(1) |
| insert / push | O(log n) |
| extract-min / pop | O(log n) |
| decrease-key (with index) | O(log n) |
| increase-key (with index) | O(log n) |
| delete-by-index | O(log n) |
| search (no index) | O(n) |
| build-heap (heapify n elements) | **O(n)** |
| heap sort | O(n log n) |


In [ ]:
# Hand-rolled min-heap (interview-style implementation)
class MinHeap:
    def __init__(self, data=None):
        self.heap = list(data) if data else []
        if data:
            self._build()

    @staticmethod
    def parent(i): return (i - 1) // 2
    @staticmethod
    def left(i):   return 2 * i + 1
    @staticmethod
    def right(i):  return 2 * i + 2

    def peek(self):
        return self.heap[0] if self.heap else None

    def push(self, x):                                # O(log n)
        self.heap.append(x)
        self._heapify_up(len(self.heap) - 1)

    def pop(self):                                    # O(log n)
        if not self.heap: return None
        if len(self.heap) == 1: return self.heap.pop()
        root = self.heap[0]
        self.heap[0] = self.heap.pop()                # move last to root, then sift down
        self._heapify_down(0)
        return root

    def _heapify_up(self, i):
        while i > 0 and self.heap[i] < self.heap[self.parent(i)]:
            self.heap[i], self.heap[self.parent(i)] = self.heap[self.parent(i)], self.heap[i]
            i = self.parent(i)

    def _heapify_down(self, i):
        n = len(self.heap)
        while True:
            smallest = i
            l, r = self.left(i), self.right(i)
            if l < n and self.heap[l] < self.heap[smallest]: smallest = l
            if r < n and self.heap[r] < self.heap[smallest]: smallest = r
            if smallest == i: return
            self.heap[i], self.heap[smallest] = self.heap[smallest], self.heap[i]
            i = smallest

    def _build(self):                                  # O(n) build from arbitrary array
        for i in range(len(self.heap) // 2 - 1, -1, -1):
            self._heapify_down(i)

    def __len__(self): return len(self.heap)

# Test build O(n), push O(log n), pop O(log n)
h = MinHeap([9, 4, 7, 1, 0, 3, -2, 8])
assert h.peek() == -2
assert h.pop() == -2
assert h.pop() == 0
h.push(-5)
assert h.peek() == -5
print("MinHeap class: OK")


## 10. Python's `heapq` library — your interview workhorse

**`heapq` operates on a plain Python list.** No class, no wrapping — the list IS the heap. This makes it very natural to use but also easy to mishandle (you can't accidentally treat `heap[3]` as the 4th smallest — it's just position 3 in the array).

**heapq is always a MIN-heap.** To get max-heap behavior, **negate values** on push and pop.

### The functions you'll actually use in interviews

| Function | What it does | Complexity |
|---|---|---|
| `heapq.heappush(h, x)` | Push x onto heap h | O(log n) |
| `heapq.heappop(h)` | Pop and return the smallest | O(log n) |
| `heapq.heappushpop(h, x)` | Push x and pop the smallest in one operation (faster than two calls) | O(log n) |
| `heapq.heapreplace(h, x)` | Pop the smallest, then push x (always pops first — assumes h is non-empty) | O(log n) |
| `heapq.heapify(lst)` | Turn list into a heap **in place** | **O(n)** |
| `heapq.nlargest(k, iterable, key=None)` | Top-k largest | O(n log k) |
| `heapq.nsmallest(k, iterable, key=None)` | Top-k smallest | O(n log k) |
| `heapq.merge(*iters)` | Merge multiple sorted iterables into one sorted iterator | O(n log k) |

`heap[0]` is the smallest (the peek operation, O(1)). No `peek()` function — just index 0.


In [ ]:
import heapq

h = [5, 3, 8, 1, 9]
heapq.heapify(h)                      # O(n)
print("after heapify (heap-ordered, not sorted):", h)
print("min:", h[0])                   # peek — O(1)

heapq.heappush(h, 2)                  # O(log n)
print("after push 2:", h)

print("pop:", heapq.heappop(h))       # 1
print("pop:", heapq.heappop(h))       # 2

# heappushpop — single-step pushpop (faster than separate calls)
print("pushpop(10):", heapq.heappushpop(h, 10))   # pops the current min
print("pushpop(0):", heapq.heappushpop(h, 0))     # 0 is the new min — returns 0 immediately

# heapreplace — pop FIRST then push (different from pushpop!)
print("replace(7):", heapq.heapreplace(h, 7))     # pops current min, pushes 7


### The max-heap trick: negate everything

Python's heapq has no max-heap mode. To simulate one, push negated values; the smallest negated value is the largest original. Negate back on pop.


In [ ]:
def max_heap_demo():
    h = []
    for x in [5, 3, 8, 1, 9]:
        heapq.heappush(h, -x)               # push negated
    largest = -heapq.heappop(h)             # negate back
    return largest

assert max_heap_demo() == 9

# Or use nlargest / nsmallest for one-shot top-K
arr = [5, 3, 8, 1, 9, 2, 7]
print("3 largest:", heapq.nlargest(3, arr))    # [9, 8, 7]
print("3 smallest:", heapq.nsmallest(3, arr))  # [1, 2, 3]


### Heaps with custom priorities — the tuple trick

The most common interview need: heap by **secondary** priority (not the values themselves but some computed key — frequency, distance, etc.).

**Pattern 1: tuples.** `heapq` compares tuples lexicographically. Push `(priority, value)`. Tiebreakers walk to the right element of the tuple.


In [ ]:
# Heap by frequency: smallest frequency wins
h = []
heapq.heappush(h, (3, 'banana'))
heapq.heappush(h, (1, 'apple'))
heapq.heappush(h, (2, 'cherry'))
print("smallest by freq:", heapq.heappop(h))   # (1, 'apple')

# What if priorities tie and values aren't comparable?
# Use a sequence number tiebreaker to enforce a deterministic order.
import itertools
counter = itertools.count()                    # generates 0,1,2,...
h = []
heapq.heappush(h, (3, next(counter), {"id": 1}))   # dicts aren't < comparable
heapq.heappush(h, (3, next(counter), {"id": 2}))
heapq.heappush(h, (1, next(counter), {"id": 3}))
print("with sequence tiebreaker:", heapq.heappop(h))


**The `queue.PriorityQueue` class also exists**, but it's thread-safe with internal locking → **strictly slower** than `heapq`. **Don't use it in interviews unless you specifically need thread safety.** Mention you know it exists but defaulted to `heapq` for performance.

### Custom comparator via `__lt__`
If you have a custom class, define `__lt__` and `heapq` will use it. Don't define `__eq__` unless needed — it makes the dict-based deduplication of identical objects different.


In [ ]:
class Task:
    def __init__(self, priority, name):
        self.priority = priority
        self.name = name
    def __lt__(self, other):                       # heapq uses this
        return self.priority < other.priority
    def __repr__(self): return f"Task({self.priority},{self.name})"

h = []
heapq.heappush(h, Task(3, 'B'))
heapq.heappush(h, Task(1, 'A'))
heapq.heappush(h, Task(2, 'C'))
print("class with __lt__:", heapq.heappop(h))    # Task(1, 'A')


## 11. Heap sort

**Algorithm.**
1. Build a max-heap from the array — O(n).
2. Repeatedly swap heap[0] with heap[end], shrink heap by 1, heapify-down the new root → element at `end` is now finalized as the largest. Repeat n-1 times.

After n-1 iterations, the array is sorted ascending **in place.**

**Time O(n log n)** — n calls to heapify-down at O(log n) each. **Space O(1)** — fully in place.

**Stability:** Not stable. (Swapping for the largest into position can disrupt order of equal elements.)

### Heap sort vs the alternatives

| Algo | Time | Space | Stable | Notes |
|------|------|-------|--------|-------|
| Heap sort | O(n log n) | **O(1)** | No | Worst-case guaranteed; great when memory is tight |
| Merge sort | O(n log n) | O(n) | Yes | Worst-case guaranteed; needs extra space |
| Quicksort | O(n log n) avg, O(n²) worst | O(log n) stack | No | Fastest in practice; pivot strategy matters |
| Python's `sort` (Timsort) | O(n log n) | O(n) | Yes | Uses runs + binary insertion + merge; what `sorted()` actually does |

**Why heap sort isn't the default sort in any major language despite its great worst case:** because cache behavior is awful — every heapify-down touches widely-spaced array positions. Quicksort and Timsort win in practice due to better cache locality, even though their theoretical worst case is worse.

Mention this in an interview when asked "why isn't heap sort always preferred?" That's a classic "deeper systems thinking" probe.


In [ ]:
def heap_sort(arr):
    n = len(arr)

    def max_heapify_down(end, i):
        while True:
            largest = i
            l, r = 2*i + 1, 2*i + 2
            if l < end and arr[l] > arr[largest]: largest = l
            if r < end and arr[r] > arr[largest]: largest = r
            if largest == i: return
            arr[i], arr[largest] = arr[largest], arr[i]
            i = largest

    # 1) Build max-heap in O(n)
    for i in range(n // 2 - 1, -1, -1):
        max_heapify_down(n, i)
    # 2) Repeatedly extract max
    for end in range(n - 1, 0, -1):
        arr[0], arr[end] = arr[end], arr[0]
        max_heapify_down(end, 0)
    return arr

assert heap_sort([9, 4, 7, 1, 0, 3, -2, 8]) == [-2, 0, 1, 3, 4, 7, 8, 9]
assert heap_sort([]) == []
assert heap_sort([1]) == [1]
print("heap_sort: OK — O(n log n) time, O(1) space, not stable")


## 12. Heap pattern 1 — Top-K

**Signal.** "K largest / smallest / most frequent / closest" elements out of N.

**Naive.** Sort everything → O(n log n). Way too much work if K << N.

**Heap trick.** Maintain a heap of **exactly size K**. After processing all N elements, the heap contains the answer.
- **K largest** → use a **min-heap** of size K (the root is the K-th largest; anything smaller gets evicted).
- **K smallest** → use a **max-heap** of size K.

**Time O(n log K), Space O(K).** When K << N, this beats sorting by an order of magnitude.

**Why min-heap for K-largest?** Counterintuitive at first. You're keeping the K "best" you've seen, and the smallest of those K is at the root. When a new element comes in, compare to root: if larger, evict root, push new. The root is always "the worst of the K best" — i.e., the Kth largest after processing all N.

### 12.1 Kth Largest Element (LC 215)

**Min-heap of size K.** Push first K; for each later element, if larger than root, replace root.

**Bonus mention:** Quickselect is O(n) average (vs O(n log K) heap), but O(n²) worst case. Interview answer: "heap is O(n log K) guaranteed; quickselect is O(n) average but has worst case; in practice heap is the cleaner answer unless K is very small or the interviewer asks for O(n)."


In [ ]:
def find_kth_largest(nums, k):
    h = nums[:k]
    heapq.heapify(h)                            # O(k)
    for x in nums[k:]:                          # O((n-k) log k)
        if x > h[0]:
            heapq.heapreplace(h, x)             # one-op pop+push
    return h[0]

assert find_kth_largest([3,2,1,5,6,4], 2) == 5
assert find_kth_largest([3,2,3,1,2,4,5,5,6], 4) == 4
print("find_kth_largest: OK — O(n log k)")


### 12.2 Top K Frequent Elements (LC 347)

Two-step pattern that repeats across this whole section:
1. **Build a frequency map** in O(n).
2. **Heap of size k by frequency** in O(n log k).

`Counter` + `nlargest` works as a one-liner using a `key`.


In [ ]:
from collections import Counter

def top_k_frequent(nums, k):
    freq = Counter(nums)
    return [x for x, _ in heapq.nlargest(k, freq.items(), key=lambda kv: kv[1])]

assert sorted(top_k_frequent([1,1,1,2,2,3], 2)) == [1, 2]
assert top_k_frequent([1], 1) == [1]
print("top_k_frequent: OK")

# Alternative for the special case: bucket sort by frequency → O(n).
# Bucket index = frequency. Walk buckets from high to low, collect k.
def top_k_frequent_bucket(nums, k):
    freq = Counter(nums)
    buckets = [[] for _ in range(len(nums) + 1)]
    for num, f in freq.items():
        buckets[f].append(num)
    out = []
    for f in range(len(buckets) - 1, 0, -1):
        for num in buckets[f]:
            out.append(num)
            if len(out) == k: return out
    return out

assert sorted(top_k_frequent_bucket([1,1,1,2,2,3], 2)) == [1, 2]
print("top_k_frequent_bucket: OK — O(n) when k is small but bucket idea worth knowing")


### 12.3 K Closest Points to Origin (LC 973)

Distance = `x² + y²`. We want the K closest → K smallest distances → **max-heap of size K** (negate distance).


In [ ]:
def k_closest(points, k):
    h = []
    for x, y in points:
        d = -(x*x + y*y)                        # negate for max-heap-like behavior
        if len(h) < k:
            heapq.heappush(h, (d, x, y))
        elif d > h[0][0]:                       # the current point is closer
            heapq.heapreplace(h, (d, x, y))
    return [[x, y] for _, x, y in h]

result = k_closest([[1,3],[-2,2]], 1)
assert result == [[-2, 2]]
result = k_closest([[3,3],[5,-1],[-2,4]], 2)
assert sorted(result) == sorted([[3,3],[-2,4]])
print("k_closest: OK")


### 12.4 Sort Characters by Frequency (LC 451)

Same pattern — heap by frequency, build the output.


In [ ]:
def frequency_sort(s):
    freq = Counter(s)
    # nlargest with key works directly; no need to roll a heap manually
    items = heapq.nlargest(len(freq), freq.items(), key=lambda kv: kv[1])
    return ''.join(ch * f for ch, f in items)

# Multiple valid outputs — verify by frequency, not exact string
out = frequency_sort("tree")
assert Counter(out) == Counter("tree")
assert out.count('e') == 2 and out.index('e') < out.index('t') and out.index('e') < out.index('r')
print("frequency_sort: OK")


### Top-K practice problems

| Concept | LeetCode |
|---------|----------|
| Kth Largest Element in Array | LC 215 |
| Top K Frequent Elements | LC 347 |
| Top K Frequent Words | LC 692 |
| K Closest Points to Origin | LC 973 |
| Sort Characters By Frequency | LC 451 |
| Kth Largest Element in a Stream | LC 703 |
| Last Stone Weight | LC 1046 |
| Find K Pairs with Smallest Sums | LC 373 |
| Maximum Subsequence Score | LC 2542 |


## 13. Heap pattern 2 — K-way merge

**Signal.** "K sorted lists / arrays / streams — merge them" or "find the smallest range covering one element from each of K lists."

**Trick.** Push the first element of every list into a heap, each tagged with which list it came from. Pop the smallest, append to output, then push the next element from the same source list. Repeat.

**Time O(N log K)** where N is total elements, **Space O(K).**

This dramatically beats the brute alternatives:
- Concatenate and sort everything: O(N log N).
- Merge pairs one by one: O(N·K) in the worst case.

### 13.1 Merge K Sorted Lists / Arrays (LC 23)

The classic. Stream from K sources, always pulling the global minimum in O(log K).


In [ ]:
def merge_k_sorted_arrays(arrays):
    h = []
    # Initialize: push (value, list_index, element_index) for each non-empty list
    for li, arr in enumerate(arrays):
        if arr:
            heapq.heappush(h, (arr[0], li, 0))
    out = []
    while h:
        val, li, ei = heapq.heappop(h)
        out.append(val)
        if ei + 1 < len(arrays[li]):
            nxt = arrays[li][ei + 1]
            heapq.heappush(h, (nxt, li, ei + 1))
    return out

assert merge_k_sorted_arrays([[10, 20, 30], [5, 15], [1, 9, 11, 18]]) == [1, 5, 9, 10, 11, 15, 18, 20, 30]
assert merge_k_sorted_arrays([[], [1], []]) == [1]
print("merge_k_sorted_arrays: OK — O(N log K), O(K) space")


### 13.2 Smallest Range Covering Elements from K Lists (LC 632)

Find the smallest range `[a, b]` such that at least one number from each of the K lists is in the range.

**Trick.** Maintain a min-heap with the current "frontier" — one element from each list. Also track the current max in the frontier. Each iteration, pop the smallest (which is `a`); `b` is the current max. Update best range. Then advance the popped list one position and push the new element (updating max).

**Why this works.** At every step, the heap contains exactly one element from each list, so the heap min and max define a valid covering range. By advancing the min, we're "trying the next smallest possible window" — guaranteed to find the optimum.

**Time O(N log K) where N is total elements. Space O(K).**


In [ ]:
def smallest_range(nums):
    h = []
    cur_max = float('-inf')
    for li, lst in enumerate(nums):
        heapq.heappush(h, (lst[0], li, 0))
        cur_max = max(cur_max, lst[0])
    best_lo, best_hi = h[0][0], cur_max
    while True:
        v, li, ei = heapq.heappop(h)
        if cur_max - v < best_hi - best_lo:
            best_lo, best_hi = v, cur_max
        if ei + 1 == len(nums[li]):
            break                                       # the popped list is exhausted
        nxt = nums[li][ei + 1]
        cur_max = max(cur_max, nxt)
        heapq.heappush(h, (nxt, li, ei + 1))
    return [best_lo, best_hi]

assert smallest_range([[4,10,15,24,26],[0,9,12,20],[5,18,22,30]]) == [20,24]
print("smallest_range: OK")


### 13.3 Find K Pairs with Smallest Sums (LC 373)

Given two sorted arrays, find the K pairs `(a, b)` with smallest `a + b`.

**Trick.** Use a min-heap. Seed with `(nums1[0] + nums2[j], 0, j)` for j = 0..min(k, len2). Each pop yields one of the K smallest sums; push the next sum from the same row.

Why seeded by row? Because sums are sorted within each row (since nums2 is sorted), and we want the global min across all rows.

**Time O(K log K), Space O(K).**


In [ ]:
def k_smallest_pairs(nums1, nums2, k):
    if not nums1 or not nums2: return []
    h = []
    # Push (sum, i, j) for one row — seed with all of nums1 paired with nums2[0]
    for i in range(min(k, len(nums1))):
        heapq.heappush(h, (nums1[i] + nums2[0], i, 0))
    out = []
    while h and len(out) < k:
        s, i, j = heapq.heappop(h)
        out.append([nums1[i], nums2[j]])
        if j + 1 < len(nums2):
            heapq.heappush(h, (nums1[i] + nums2[j+1], i, j+1))
    return out

assert k_smallest_pairs([1,7,11], [2,4,6], 3) == [[1,2],[1,4],[1,6]]
print("k_smallest_pairs: OK")


### K-way merge practice problems

| Concept | LeetCode |
|---------|----------|
| Merge K Sorted Lists | LC 23 |
| Merge K Sorted Arrays | GFG classic |
| Find K Pairs with Smallest Sums | LC 373 |
| Smallest Range Covering Elements from K Lists | LC 632 |
| Kth Smallest Element in a Sorted Matrix | LC 378 (K-way merge framing) |
| Merge Sorted Array (2 arrays, in-place) | LC 88 |
| Ugly Number II | LC 264 (3-way merge of multiples) |


## 14. Heap pattern 3 — Two heaps

**Signal.** "Running median" or "median of a sliding window" or any "middle-of-stream" query.

**Trick.** Maintain TWO heaps:
- **`lo`** — a **max-heap** holding the smaller half of seen values. Its root is the largest of the smaller half.
- **`hi`** — a **min-heap** holding the larger half. Its root is the smallest of the larger half.

**Invariants:**
- Every element in `lo` is ≤ every element in `hi`.
- Sizes differ by at most 1. Convention: `len(lo) == len(hi)` or `len(lo) == len(hi) + 1`.

**Median:**
- If `len(lo) > len(hi)` → median is `lo[0]`.
- If they're equal → median is `(lo[0] + hi[0]) / 2`.

### 14.1 Find Median from Data Stream (LC 295)

**Insert protocol** (the trickiest part — get this right every time):
1. Push the new number to `lo` (as `-num` since `lo` is max-heap via negation).
2. Move `lo`'s top to `hi`. This guarantees `lo`'s top ≤ `hi`'s top.
3. If `hi` has become larger than `lo`, move `hi`'s top back to `lo` to keep sizes balanced.

**Each insert is O(log n); each query is O(1).**


In [ ]:
class MedianFinder:
    def __init__(self):
        self.lo = []         # max-heap (store negated)
        self.hi = []         # min-heap

    def add_num(self, num):
        # 1) push into lo (negated)
        heapq.heappush(self.lo, -num)
        # 2) move lo's top to hi
        heapq.heappush(self.hi, -heapq.heappop(self.lo))
        # 3) balance: lo should have equal or one more
        if len(self.hi) > len(self.lo):
            heapq.heappush(self.lo, -heapq.heappop(self.hi))

    def find_median(self):
        if len(self.lo) > len(self.hi):
            return -self.lo[0]
        return (-self.lo[0] + self.hi[0]) / 2

mf = MedianFinder()
mf.add_num(1)
mf.add_num(2)
assert mf.find_median() == 1.5
mf.add_num(3)
assert mf.find_median() == 2
mf.add_num(4)
assert mf.find_median() == 2.5
print("MedianFinder: OK")


### 14.2 Median in a Sliding Window (LC 480)

Two-heap median + **sliding window** means we also need to remove elements as the window slides — but heaps don't support delete-by-value in O(log n) directly.

**Two strategies:**

1. **Lazy deletion.** Mark elements as "to be removed" in a hash set. When you'd peek, first pop all lazily-deleted elements from the top. Amortized O(log n) but ugly.

2. **`SortedList` from `sortedcontainers`.** A balanced BST-like structure with O(log n) insert/delete by value AND O(log n) index access. Use `bisect_left` to find the position to delete; index `k//2` and `k//2 - 1` give the median elements. **Much cleaner.** This is the accepted "Python idiomatic" solution.

In an interview, mention both — and note that `sortedcontainers` is third-party (though it's available on LeetCode). For "pure stdlib," do the lazy-deletion two-heap version.

### 14.3 IPO (LC 502) — different "two heap" variant

`k` projects to invest in. Each has profit and capital requirement. Start with `w` capital. After each project completes, your capital grows by the profit. Pick k projects to maximize final capital.

**Strategy.** Two structures:
- **Min-heap** by capital (projects you can't afford yet).
- **Max-heap** by profit (projects you CAN afford, sorted by best profit).

Each round: move all newly-affordable projects from the min-heap to the max-heap. Pop the max-profit one. Repeat k times.

**Time O((n + k) log n).**


In [ ]:
def find_maximized_capital(k, w, profits, capital):
    # Sort by capital ascending (or use a min-heap by capital)
    min_cap = sorted(zip(capital, profits))           # by capital ascending
    max_prof = []                                      # affordable profits, negated
    i = 0
    for _ in range(k):
        # Move all now-affordable into max_prof
        while i < len(min_cap) and min_cap[i][0] <= w:
            heapq.heappush(max_prof, -min_cap[i][1])
            i += 1
        if not max_prof:                              # nothing affordable
            break
        w += -heapq.heappop(max_prof)
    return w

assert find_maximized_capital(2, 0, [1,2,3], [0,1,1]) == 4
assert find_maximized_capital(3, 0, [1,2,3], [0,1,2]) == 6
print("find_maximized_capital: OK")


### Two-heap / median practice problems

| Concept | LeetCode |
|---------|----------|
| Find Median from Data Stream | LC 295 |
| Sliding Window Median | LC 480 |
| Find Right Interval | LC 436 |
| IPO | LC 502 |
| Maximum Number in Sliding Window (median variant) | — |


## 15. Heap pattern 4 — Scheduling and Greedy

**Signal.** "Tasks with cooldown / deadline / capacity" or "rooms / lanes / processors."

The heap orders "candidates I can pick next, by best priority right now." After picking, you re-insert candidates that have become available.

### 15.1 Task Scheduler (LC 621)

n tasks, identical tasks need `cooldown` ticks between them. Find the minimum total ticks.

**Heap approach.** Max-heap by current remaining frequency. Each tick, try to pick the most-frequent available task. Put the just-used task into a cooldown queue with its "ready-again" time.

**Math approach (often easier).** Maximum frequency = `f_max`. There are `most_frequent_count` tasks tied for max. The answer is:
```
max( len(tasks), (f_max - 1) * (n + 1) + most_frequent_count )
```
The heap solution generalizes; the math doesn't. Memorize both.


In [ ]:
def least_interval(tasks, n):
    if n == 0: return len(tasks)
    freq = Counter(tasks)
    max_freq = max(freq.values())
    tied = sum(1 for f in freq.values() if f == max_freq)
    return max(len(tasks), (max_freq - 1) * (n + 1) + tied)

assert least_interval(["A","A","A","B","B","B"], 2) == 8
assert least_interval(["A","C","A","B","D","B"], 1) == 6
assert least_interval(["A","A","A","B","B","B"], 0) == 6
print("least_interval (math approach): OK — O(n)")


In [ ]:
# Heap-based version — works even when math gets complex
def least_interval_heap(tasks, n):
    freq = Counter(tasks)
    h = [-c for c in freq.values()]
    heapq.heapify(h)
    time = 0
    while h:
        cooldown = []                            # tasks waiting to come back
        for _ in range(n + 1):
            if h:
                c = heapq.heappop(h)
                if c + 1 < 0:                    # still have remaining count
                    cooldown.append(c + 1)
            time += 1
            if not h and not cooldown:           # all done
                break
        for c in cooldown:
            heapq.heappush(h, c)
    return time

assert least_interval_heap(["A","A","A","B","B","B"], 2) == 8
print("least_interval_heap: OK — generalizable")


### 15.2 Meeting Rooms II (LC 253) — min rooms needed

Given meeting intervals, return the minimum number of rooms needed.

**Heap approach.** Sort by start time. For each meeting, if the room with the earliest end-time (heap top) is free (end ≤ current start), reuse it (pop, push the new end). Else, allocate a new room (push). The heap size at the end is the answer.

**Time O(n log n) for sort + O(n log n) for heap ops.**


In [ ]:
def min_meeting_rooms(intervals):
    if not intervals: return 0
    intervals.sort(key=lambda x: x[0])
    h = []                                       # heap of room end-times
    for start, end in intervals:
        if h and h[0] <= start:                  # earliest-ending room is free
            heapq.heapreplace(h, end)            # reuse: replace its end time
        else:
            heapq.heappush(h, end)               # need a new room
    return len(h)

assert min_meeting_rooms([[0,30],[5,10],[15,20]]) == 2
assert min_meeting_rooms([[7,10],[2,4]]) == 1
print("min_meeting_rooms: OK")


### 15.3 Reorganize String (LC 767)

Rearrange a string so no two adjacent chars are equal. If impossible, return "".

**Strategy.** Max-heap by remaining frequency. Pop the most frequent (must place); also pop the second-most-frequent (will be the alternation). Place both, decrement, and push back if still have count.

**Feasibility check.** If any single char appears > (n+1) // 2 times, return "".


In [ ]:
def reorganize_string(s):
    n = len(s)
    freq = Counter(s)
    if max(freq.values()) > (n + 1) // 2:
        return ""
    h = [(-c, ch) for ch, c in freq.items()]
    heapq.heapify(h)
    out = []
    while len(h) >= 2:
        c1, ch1 = heapq.heappop(h)
        c2, ch2 = heapq.heappop(h)
        out.append(ch1); out.append(ch2)
        if c1 + 1 < 0: heapq.heappush(h, (c1 + 1, ch1))
        if c2 + 1 < 0: heapq.heappush(h, (c2 + 1, ch2))
    if h:
        c, ch = h[0]
        out.append(ch)
    return ''.join(out)

result = reorganize_string("aab")
assert result == "aba"
result = reorganize_string("aaab")
assert result == ""
result = reorganize_string("aabb")
# Multiple valid: "abab", "baba", etc. — just verify the constraint
for i in range(1, len(result)):
    assert result[i] != result[i-1]
print("reorganize_string: OK")


### Scheduling practice problems

| Concept | LeetCode |
|---------|----------|
| Task Scheduler | LC 621 |
| Meeting Rooms II | LC 253 |
| Reorganize String | LC 767 |
| Reorganize K Distance Apart | LC 358 |
| Minimum Number of CPU Tasks | LC 1834 (Single-Threaded CPU) |
| Maximum Performance of a Team | LC 1383 |
| Furthest Building You Can Reach | LC 1642 |


## 16. Heap pattern 5 — Kth element in a sorted structure

**Signal.** "Kth smallest in a sorted matrix" or "Kth-ugly number" — a structure where you can enumerate candidates in sorted order using a heap, popping K times.

### 16.1 Kth Smallest Element in a Sorted Matrix (LC 378)

An n×n matrix where each row AND each column is sorted ascending. Find the kth smallest element.

**Heap approach (K-way merge).** Each row is a sorted list. Run k-way merge until you've extracted k elements.

**Better: binary search on value** — O(n log(max-min)) — but the heap version is the textbook teaching example.

**Time (heap): O(K log min(K, N)), Space O(min(K, N)).**


In [ ]:
def kth_smallest_matrix(matrix, k):
    n = len(matrix)
    h = []
    for r in range(min(k, n)):
        heapq.heappush(h, (matrix[r][0], r, 0))
    for _ in range(k - 1):
        v, r, c = heapq.heappop(h)
        if c + 1 < n:
            heapq.heappush(h, (matrix[r][c+1], r, c+1))
    return h[0][0]

mat = [[1,5,9],[10,11,13],[12,13,15]]
assert kth_smallest_matrix(mat, 8) == 13
assert kth_smallest_matrix([[-5]], 1) == -5
print("kth_smallest_matrix: OK")


### 16.2 Ugly Number II (LC 264)

Find the nth ugly number (only prime factors 2, 3, 5).

**Heap approach.** Push 1. Pop smallest, push 2x, 3x, 5x of it (deduplicated). nth pop is the answer.

**Better: 3-pointer DP** — O(n) — but the heap version is intuitive.


In [ ]:
def nth_ugly(n):
    h = [1]
    seen = {1}
    ugly = 0
    for _ in range(n):
        ugly = heapq.heappop(h)
        for p in (2, 3, 5):
            nxt = ugly * p
            if nxt not in seen:
                seen.add(nxt)
                heapq.heappush(h, nxt)
    return ugly

assert nth_ugly(10) == 12
assert nth_ugly(1) == 1
print("nth_ugly: OK — heap version O(n log n)")


### 16.3 K-th Smallest Prime Fraction (LC 786)

Given a sorted array of primes-plus-1, find the kth smallest fraction `a/b` formed from two elements.

Same idea: min-heap of fractions, advance through.

### Kth-in-sorted-structure practice problems

| Concept | LeetCode |
|---------|----------|
| Kth Smallest in Sorted Matrix | LC 378 |
| Ugly Number II | LC 264 |
| Super Ugly Number | LC 313 |
| Kth Smallest Prime Fraction | LC 786 |
| K-th Smallest Number in Multiplication Table | LC 668 |
| Find K-th Smallest Pair Distance | LC 719 |


## 17. Decision matrix — when to reach for which

These are the signals interviewers drop intentionally:

| Phrase in problem | First instinct |
|---|---|
| "matching", "balanced", "valid" parens / tags | Stack |
| "expression evaluation", "calculator", "polish notation" | Stack |
| "undo / redo" | Stack |
| "next/previous greater/smaller" | Monotonic stack |
| "largest rectangle", "histogram" | Monotonic stack |
| "trapping rain water" | Monotonic stack OR two-pointers (both valid) |
| "daily temperatures", "stock span" | Monotonic stack |
| "shortest path in unweighted graph", "minimum steps", "spread out from sources" | BFS → Queue |
| "level by level" | Queue + level snapshot |
| "first negative / first X in every window of size k" | Queue / deque |
| "circular tour" | Queue simulation, but think O(n) one-pass first |
| "max/min in every sliding window of size k" | Monotonic deque |
| "DP transition with sliding-window range" | DP + monotonic deque |
| "Kth largest / smallest / closest / most frequent" | Heap of size K |
| "merge K sorted ..." | Min-heap (K-way merge) |
| "running / streaming median" | Two heaps |
| "task scheduler with cooldown / capacity / deadline" | Heap (scheduling) |
| "meeting rooms / min processors / lanes needed" | Sort + min-heap |
| "rearrange / reorganize string" | Max-heap by frequency |
| "Kth smallest in sorted matrix" | Heap K-way merge OR binary search on value |

### When NOT to reach for a heap

- **You only need ONE max or min, once** — a single pass O(n) with a tracker variable beats O(n log n) heap.
- **You need ALL elements sorted** — just sort, O(n log n) once.
- **You need to remove arbitrary elements** — heap doesn't support this in O(log n) without an index map. Consider `SortedList` (sortedcontainers) or a balanced BST.
- **Static dataset where K-th is a one-off query** — quickselect O(n) average is better than heap O(n log K).


## 18. Synthesis — complexity table, narration prep, common bugs

### 18.1 Full complexity reference

| Operation | Stack | Queue (deque) | Deque | MinHeap | Sorted array | HashMap |
|---|---|---|---|---|---|---|
| Peek extreme end | O(1) | O(1) front | O(1) both ends | O(1) | O(1) | — |
| Push / insert | O(1) | O(1) | O(1) | O(log n) | O(n) | O(1) |
| Pop / extract extreme | O(1) | O(1) | O(1) | O(log n) | O(1) | — |
| Find arbitrary | O(n) | O(n) | O(n) | O(n) | O(log n) | O(1) |
| Build from n elements | O(n) | O(n) | O(n) | **O(n)** | O(n log n) | O(n) |
| Delete by value | O(n) | O(n) | O(n) | O(n) without index | O(n) | O(1) |

### 18.2 Algorithm complexity table

| Problem | Time | Space | Optimization made |
|---|---|---|---|
| Valid parentheses | O(n) | O(n) | Stack records "what we expect next" |
| Min Stack (aux stack) | O(1) per op | O(n) | Parallel running-min stack |
| Min Stack (delta encoding) | O(1) per op | O(1) extra | Encode "minimum overwrite" inline |
| Infix → Postfix | O(n) | O(n) | Stack holds pending operators by precedence |
| Postfix evaluation | O(n) | O(n) | Stack of operands |
| Next greater element | O(n) | O(n) | Each index pushed/popped at most once |
| Daily temperatures | O(n) | O(n) | Same as NGE, with index distance |
| Stock span | O(n) amortized | O(n) | Previous-greater via decreasing stack |
| Largest rectangle in histogram | O(n) | O(n) | One-pass increasing stack with virtual sentinel |
| Trapping rain water (stack) | O(n) | O(n) | Same approach; two-pointer is O(1) space |
| Sum of subarray minimums | O(n) | O(n) | Contribution via PSE + NSE |
| BFS / level order | O(V + E) | O(V) | Queue ensures FIFO exploration |
| Rotting oranges (multi-source BFS) | O(m·n) | O(m·n) | All sources in initial queue |
| First negative in window | O(n) | O(k) | Deque of negative-element indices |
| Circular tour (gas station) | O(n) | O(1) | Surplus argument removes the queue |
| Sliding window max/min | O(n) | O(k) | Monotonic deque — each index in/out at most once |
| Constrained subseq sum | O(n) | O(k) | DP + monotonic deque |
| Heapify n elements | O(n) | O(1) | Bottom-up; sum of heights is O(n) |
| Heap sort | O(n log n) | O(1) | In-place but not stable |
| Kth largest (heap) | O(n log k) | O(k) | Heap of size k, not n |
| Top K frequent | O(n log k) | O(n + k) | Counter + heap of size k |
| K Closest Points | O(n log k) | O(k) | Max-heap of size k by distance |
| Merge K sorted | O(N log K) | O(K) | Heap always holds one frontier per list |
| Smallest range from K lists | O(N log K) | O(K) | Heap of frontiers + running max |
| Median from stream | O(log n) insert, O(1) query | O(n) | Two heaps balanced |
| Sliding window median | O(n log n) | O(k) | Two heaps with lazy delete, or SortedList |
| Task Scheduler (heap) | O(n log u) where u = unique tasks | O(u) | Heap by remaining frequency |
| Meeting rooms II | O(n log n) | O(n) | Sort by start + heap by end |
| Reorganize string | O(n log u) | O(u) | Max-heap by frequency + 2-at-a-time placement |
| Kth smallest in matrix | O(k log min(k,n)) | O(min(k,n)) | K-way merge of rows |

### 18.3 Interview narration — verbatim answers

**Q: Why use `collections.deque` instead of a list for queues?**
> Because `list.pop(0)` is O(n) — it shifts every remaining element left. In a BFS loop this turns O(V+E) into O(V²+VE), which TLEs on large inputs. `collections.deque` is a doubly-linked list of fixed-size blocks; both ends are O(1). Always use deque for queues in Python.

**Q: Why is monotonic stack O(n)?**
> Each element is pushed onto the stack at most once and popped at most once. So across n outer iterations, the total work in the inner while loops sums to at most 2n. Total time is O(n) amortized, with O(n) space for the stack.

**Q: Why is build-heap O(n) and not O(n log n)?**
> Because the cost of `heapify-down` at a node depends on the node's height — its distance to the deepest leaf — not its depth. In a complete tree of size n, there are roughly n/2 leaves at height 0, n/4 nodes at height 1, n/8 nodes at height 2, and so on. Summing n / 2^(h+1) × h over all heights gives a series that converges to O(n). The naive "n inserts at O(log n) each" gives O(n log n), but bottom-up heapify is strictly better.

**Q: Why use a min-heap of size K for the K largest elements?**
> Because the smallest of the K best you've kept is at the root. When a new element arrives, you compare it against the root in O(1). If larger, the root is no longer in the top K — pop it, push the new one. The K largest at the end are exactly the heap's contents, and the root is the Kth largest. Total cost: O(N log K), much better than O(N log N) sort when K << N.

**Q: Why two heaps for running median?**
> One heap (max-heap) holds the smaller half, the other (min-heap) holds the larger half. The median is either the top of whichever heap is larger, or the average of the two tops if sizes are equal. Insert is O(log n) — push, then move tops across to enforce the invariant that everything in lo ≤ everything in hi. Median query is O(1). Compared to inserting into a sorted list — O(n) per insert — this is dramatically faster on a stream.

**Q: For sliding window max, why a monotonic deque and not a heap?**
> Because we need to remove **out-of-window** elements from the **front** in O(1). A heap supports O(log n) extraction of the max but not of an arbitrary element. The monotonic deque maintains decreasing order from front to back; the front is always the current window max, and stale indices are popped from the front in O(1).

**Q: Why is `heapq.PriorityQueue` slower than `heapq`?**
> Because `PriorityQueue` is thread-safe with internal locking. Every `put` and `get` acquires a lock. For a single-threaded interview problem, that's pure overhead. `heapq` operates on a plain list and has no locking, so it's strictly faster. I'd only use `PriorityQueue` for actual multi-threaded code.

**Q: What's the difference between `heappushpop` and `heapreplace`?**
> `heappushpop(h, x)` pushes x then pops — if x is smaller than the current min, x is returned immediately without modifying the heap. `heapreplace(h, x)` pops first then pushes — it always pops the current min. So heapreplace assumes the heap is non-empty and is the right choice when you've already determined you want to replace the root.

**Q: Why is Largest Rectangle in Histogram O(n) despite the nested loops?**
> Because the inner while loop pops indices off the stack, and each index is pushed at most once across the entire outer loop. So the inner loop's total work is O(n), not O(n) per outer iteration. Amortized total: O(n).

**Q: What's the trap with `int(a/b)` vs `a//b` for postfix evaluation in LeetCode?**
> `a//b` is floor division — it rounds toward negative infinity. LeetCode 150 specifies truncation toward zero. So `-7 // 2 == -4` is wrong; `int(-7/2) == -3` is correct. Always use `int(a/b)` for that problem.

### 18.4 Common bugs and traps

1. **`list.pop(0)` as a queue** — O(n) per call. Use `collections.deque`.
2. **Negating for max-heap but forgetting to negate back** — push `-x`, but on pop the value is `-h[0]`.
3. **In `MinStack`, only pushing onto `min_stack` when smaller** — when popping, the main stack and the min stack get out of sync. Either push every time (with the running min) OR also push the matching count.
4. **Postfix eval with operands in wrong order** — `b = pop()` first, `a = pop()` second. `a OP b`, not `b OP a`.
5. **Postfix `int(a/b)` vs `a//b`** — LeetCode truncates toward zero, not floor. Use `int(a/b)`.
6. **Monotonic stack with `<` vs `<=`** — strict-vs-non-strict matters for duplicate handling. In contribution problems (sum of subarray mins), use strict on one side and non-strict on the other to avoid double counting.
7. **Forgetting the sentinel in Largest Rectangle** — without the trailing 0-height sentinel, bars that never see a shorter right neighbor never get processed.
8. **Largest Rectangle: storing values instead of indices** — you lose the width calculation. Always store indices.
9. **BFS without a `visited` set** — infinite loop on graphs with cycles.
10. **Marking visited when popping instead of when pushing** — same node enters the queue multiple times via different predecessors → O(V²) memory and TLE.
11. **`for _ in range(len(q))` evaluated lazily** — if `q` is a deque and `len(q)` is called inside the loop, you'll process newly-added neighbors in the same level. Snapshot the length before the inner loop, or use the `for _ in range(len(q))` pattern at the start.
12. **Monotonic deque sliding window: not evicting before processing** — if you pop the back first and then check `if q[0] <= i - k`, you might be looking at a stale index. Evict front BEFORE pop-back-while-smaller.
13. **Two-heap median balance bug** — pushing always to lo, then moving lo's top to hi unconditionally, then balancing if hi > lo. Get this 3-step sequence right.
14. **Custom comparator in heap with non-comparable secondary key** — if priorities tie and values are dicts/objects without `__lt__`, you get `TypeError`. Add a sequence-number tiebreaker (`itertools.count()`).
15. **Top-K with heap of size N instead of K** — defeats the entire purpose. Heap should be size K and you should `heapreplace` when the new element beats the root.
16. **`heapq.nlargest` / `nsmallest` with N elements** — for K close to N, just sorted. nlargest is O(n log k); for k = n it's O(n log n) anyway.
17. **Stock span / NGE with values on the stack instead of indices** — you lose position information. Always store indices.
18. **In LRU cache and similar problems, reaching for a heap** — heaps don't have O(1) deletion by key. Use OrderedDict (or doubly-linked-list + hashmap). Heap is for *priority*, not *recency*.
19. **K Closest Points using min-heap of size K** — wrong direction. Min-heap of size K keeps the K **largest**. Use max-heap (negate distance).
20. **`heapify` returns None, not the heap** — `pq = heapq.heapify(arr)` is `None`. The reference's `purchasing_maximum_items_with_given_budget` has this bug; it should be `heapq.heapify(arr); pq = arr`.

### 18.5 Final summary — the 8 master patterns

| # | Pattern | DS | Canonical | Time |
|---|---|---|---|---|
| 1 | Bracket / matching | Stack | Valid parens | O(n) |
| 2 | Expression eval | Stack | Postfix eval, calculator | O(n) |
| 3 | Min/max-stack | Stack + aux | Get-min in O(1) | O(1) per op |
| 4 | Monotonic stack | Stack | NGE, daily temps, hist rect, trap rain | O(n) |
| 5 | BFS | Queue | Shortest path unweighted, level order, rotting | O(V+E) |
| 6 | Monotonic deque | Deque | Sliding window max/min, DP transition | O(n) |
| 7 | Top-K | Heap (size K) | Kth largest, top K freq, K closest | O(n log K) |
| 8 | K-way / Two-heap / Schedule | Heap | Merge K lists, median stream, task scheduler | O(N log K) or O(log n) per op |

If you can recognize the signal in 60 seconds, name the DS, and write the template in 3 minutes — including stating WHY this DS beats the array baseline for THIS signal — you're solid.
